# Ausgrid synthetic GC+CL and gross solar — checkpointed Jupyter run

Open this notebook from the project’s `notebooks/` folder in VS Code. Put the three original Ausgrid CSVs directly in the sibling `data/` folder. The notebook locates both folders automatically and saves checkpoints and reports within the same project. Run the cells in order; rerunning after a kernel restart skips finished stages.


In [4]:
import os
import sys
from pathlib import Path

print("os.name:", os.name)
print("sys.platform:", sys.platform)
print("Python:", sys.executable)
print("/mnt/c exists:", Path("/mnt/c").exists())

os.name: posix
sys.platform: linux
Python: /usr/bin/python3
/mnt/c exists: False


In [3]:
from pathlib import Path
import os, sys, subprocess, json, time

PROJECT_ROOT = Path(
    r"C:/Users/william.intern/Desktop/Personal Project/ausgrid-synthetic/ausgrid-synthetic"
)

NOTEBOOK_DIR = PROJECT_ROOT / "notebooks"
RAW_DIR = PROJECT_ROOT / "data"

RAW_NAMES = [
    "Solar home 2010-2011.csv",
    "Solar home 2011-2012.csv",
    "Solar home 2012-2013.csv",
]

# Check project structure
assert PROJECT_ROOT.is_dir(), f"Project root not found: {PROJECT_ROOT}"

assert (PROJECT_ROOT / "src" / "ausgrid_synth" / "cli.py").is_file(), (
    f"Could not find src/ausgrid_synth/cli.py inside:\n{PROJECT_ROOT}"
)

assert NOTEBOOK_DIR.is_dir(), f"Notebook folder not found: {NOTEBOOK_DIR}"

missing = [
    name for name in RAW_NAMES
    if not (RAW_DIR / name).is_file()
]

assert not missing, (
    f"Place the three CSV files directly in:\n{RAW_DIR}\n"
    f"Missing: {missing}"
)

# Make src importable
SRC_DIR = PROJECT_ROOT / "src"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

os.environ["PYTHONPATH"] = os.pathsep.join(
    filter(
        None,
        (
            str(SRC_DIR),
            os.environ.get("PYTHONPATH")
        )
    )
)

print("Python:", sys.executable)
print("Project root:", PROJECT_ROOT)
print("Notebook folder:", NOTEBOOK_DIR)
print("CSV directory:", RAW_DIR)
print("Setup OK")

AssertionError: Project root not found: C:/Users/william.intern/Desktop/Personal Project/ausgrid-synthetic/ausgrid-synthetic

## 0. Dependencies and runner

Install `requirements.txt` **in the selected kernel environment** if needed. Set `INSTALL_REQUIREMENTS = True` once to install from this notebook, or install in your terminal. Keep it `False` when the environment is already set up. A GPU is recommended for neural training; audit and baseline work on CPU.


In [ ]:
INSTALL_REQUIREMENTS = False
if INSTALL_REQUIREMENTS:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(PROJECT_ROOT / 'requirements.txt')], check=True)
import importlib.util
missing_packages = [m for m in ('numpy', 'pandas', 'scipy', 'sklearn', 'matplotlib')
                    if importlib.util.find_spec(m) is None]
if missing_packages:
    raise ModuleNotFoundError(f'Missing {missing_packages} in {sys.executable}; install requirements.txt in this kernel.')
try:
    import torch
except ModuleNotFoundError:
    torch = None
print('PyTorch:', torch.__version__ if torch else 'not installed (needed for neural stages)')
print('CUDA GPU:', torch.cuda.get_device_name(0) if torch and torch.cuda.is_available() else 'not detected; CPU is available')
def run(stage, *flags):
    cmd = [sys.executable, '-m', 'ausgrid_synth.cli', stage, '--root', str(PROJECT_ROOT), *map(str, flags)]
    print('RUN:', ' '.join(cmd), flush=True)
    subprocess.run(cmd, check=True, env=os.environ.copy())


## Checkpoint 1 — audit and prepare

The source CSVs stay in `../data/` relative to this notebook. The prepared arrays, `audit.json`, and `split.json` are written to `../data/prepared/`. The script removes clock-change dates and invalid/estimated rows, and excludes missing CL days for households known to have CL. If you replace the CSVs or change preprocessing code, delete the old prepared output and downstream checkpoints before rerunning.


In [ ]:
prepared = PROJECT_ROOT / 'data/prepared/prepared.npz'
if not prepared.exists(): run('prepare', '--raw-dir', RAW_DIR)
audit = json.loads((PROJECT_ROOT / 'data/prepared/audit.json').read_text())
print('Usable paired days:', audit['total_usable_days'])
print('Households:', audit['split_households'])
print('Per-year exclusions:', audit['years'])
print('Train measured GG >0.02 kWh in the defined night:', audit.get('measured_night_gt_0p02_train'))


## Checkpoint 2 — statistical baseline
A paired joint model provides a non-neural comparison. It never returns original household days by resampling them.

In [ ]:
stat_model = PROJECT_ROOT / 'outputs/checkpoints/statistical.joblib'
if not stat_model.exists(): run('baseline')
print('Baseline ready:', stat_model)


## Checkpoint 3 — short neural pilot

This trains **two epochs each** of the unmasked and constrained VAE. Both write `*_latest.pt` and best-validation checkpoints. Full training resumes from those epoch checkpoints. You may run on CPU for a quick check, though a CUDA-enabled kernel is recommended for the full experiment.


In [ ]:
if torch is None:
    raise ModuleNotFoundError('Install PyTorch in this kernel or choose a kernel with PyTorch before neural training.')
if not torch.cuda.is_available():
    print('No CUDA GPU detected. The pilot can run on CPU, but full training may be slow.')
for arm in ('vae', 'vae_daylight'):
    marker = PROJECT_ROOT / 'outputs/checkpoints' / f'{arm}_seed1_status.json'
    if not marker.exists():
        started = time.time()
        run('train', '--arm', arm, '--seed', 1, '--epochs', 2)
        print(arm, 'two-epoch elapsed seconds:', round(time.time()-started))
    else: print('Pilot checkpoint already exists:', marker)


## Checkpoint 4 — full neural comparison

Six fitted models: unmasked and constrained VAE, three seeds each. The postprocessed comparison reuses the unmasked checkpoints. The best model is selected by validation loss; the latest checkpoint includes optimizer and random states for resume after a kernel interruption. Choose `EPOCHS` before viewing test results.


In [ ]:
EPOCHS = 30
for seed in (1, 2, 3):
    for arm in ('vae', 'vae_daylight'):
        status_file = PROJECT_ROOT / 'outputs/checkpoints' / f'{arm}_seed{seed}_status.json'
        status = json.loads(status_file.read_text()) if status_file.exists() else {}
        complete = status.get('target_epochs') == EPOCHS and (status.get('early_stopped') or status.get('last_epoch', 0) >= EPOCHS)
        if not complete: run('train', '--arm', arm, '--seed', seed, '--epochs', EPOCHS)
        else: print('Complete:', arm, seed)


## Checkpoint 5 — matched samples and final test
`train` contexts support the prediction test; `test` contexts support distribution checks; `privacy` samples use a common set of dates and capacities. Samples record only an internal context index for reproducible local analysis. **Do not redistribute the prepared file or context-indexed samples as a public release.** The `test` split is evaluated only here after fixing choices above.

In [ ]:
arms = [('statistical', 1)] + [(arm, seed) for seed in (1,2,3) for arm in ('vae', 'vae_post', 'vae_daylight')]
for arm, seed in arms:
    for context in ('train', 'test', 'privacy'):
        file = PROJECT_ROOT / 'outputs/samples' / f'{arm}_seed{seed}_{context}.npz'
        if not file.exists(): run('sample', '--arm', arm, '--seed', seed, '--context', context)
    report = PROJECT_ROOT / 'outputs/reports' / f'{arm}_seed{seed}.json'
    if not report.exists(): run('evaluate', '--arm', arm, '--seed', seed)
print('Completed reports:', len(list((PROJECT_ROOT/'outputs/reports').glob('*.json'))))


## Checkpoint 6 — compare and inspect
Report physical violations, daily-energy distribution error, utility versus real-trained model, and the exploratory household membership attack. Three seeds are repeated measurements of training randomness; there is one fixed household split.

In [ ]:
import pandas as pd
rows=[]
for arm, seed in arms:
    r=json.loads((PROJECT_ROOT / 'outputs/reports' / f'{arm}_seed{seed}.json').read_text())
    f,u,p=r['fidelity_and_physics'],r['utility'],r['disclosure_probe']
    rows.append({'arm':arm,'seed':seed,'night_gg_gt_0.02':f['night_solar_gt_0p02_intervals'],
                 'night_gg_kwh':f['night_solar_kwh'],
                 'load_daily_W1_kwh':f['total_consumption_daily_kwh_wasserstein'],
                 'solar_daily_W1_kwh':f['gross_generation_daily_kwh_wasserstein'],
                 'synthetic_to_real_mae':u['synthetic_train_test_real_mae'],
                 'real_to_real_mae':u['real_train_test_real_mae'],
                 'household_attack_auc':p['household_membership_auc']})
results=pd.DataFrame(rows)
display(results.round(4))
print('Read outputs/reports/*.json for full metrics and confidence intervals.')


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from ausgrid_synth.data import load_prepared
d=load_prepared(PROJECT_ROOT/'data/prepared/prepared.npz')
with np.load(PROJECT_ROOT/'outputs/samples/vae_daylight_seed1_test.npz') as z:
    generated, idx=z['x'],z['idx']
slot=np.arange(48)/2+0.25
fig, axes=plt.subplots(1,2,figsize=(12,3))
for c,title in enumerate(('GC+CL consumption','Gross solar generation')):
    axes[c].plot(slot,np.median(d['x'][idx,c],axis=0),label='real held-out')
    axes[c].plot(slot,np.median(generated[:,c],axis=0),label='synthetic',linestyle='--')
    axes[c].set(title=title,xlabel='Local hour (interval midpoint)',ylabel='kWh / half-hour')
    axes[c].legend()
plt.tight_layout();plt.show()


### Interpretation boundary
The generated units are independent days. Daily samples cannot establish week/year continuity or persistent synthetic households. The membership attack is a limited empirical probe, not a privacy certificate. Decide whether to release any generated samples only after reviewing the reports and disclosure risks.